In [ ]:
import pandas as pd
from PIL import Image
import os
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

In [2]:
seed = 42
torch.random.manual_seed(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
root_path = "/home/stefan/ioai-prep/kits/concat_img_cat_cnt"

IMG_SIZE = 224
batch_size = 64

# Data

In [3]:
class CategoryCountDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None, is_test=False):
        self.df = dataframe
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["ImagePath"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, row["SampleID"]
        else:
            label = torch.tensor(row["Label"], dtype=torch.float)
            return image, label

In [4]:
df_train = pd.read_csv(os.path.join(root_path, "train.csv"))
df_train, df_val = train_test_split(df_train, test_size=0.2, random_state=42)

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = CategoryCountDataset(df_train, root_path, transform=train_transform)
val_dataset = CategoryCountDataset(df_val, root_path, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=10)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=10)

In [5]:
# sanity check
x = next(iter(train_loader))
[y.shape for y in x]

[torch.Size([64, 3, 224, 224]), torch.Size([64])]

# Model

In [6]:
def build_model():
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    in_features = model.fc.in_features

    model.fc = nn.Sequential(
        nn.Dropout(p=0.5, inplace=True),
        nn.Linear(in_features, 256),
        nn.LeakyReLU(),
        nn.Linear(256, 1)
    )
    model = model.to(device)
    return model

# sanity check
model = build_model()
model(x[0].to(device)).shape

torch.Size([64, 1])

# Training

In [7]:
class CategoryCountLightningModule(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = build_model()
        self.learning_rate = 1e-4
        self.criterion = nn.MSELoss()

        self.train_loss = None
        self.val_loss = None
        self.val_score = None

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x).squeeze()
        loss = self.criterion(y_hat, y)

        self.log("train_loss", loss)
        self.train_loss = loss

        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x).squeeze()
        loss = self.criterion(y_hat, y)

        preds = torch.round(y_hat)
        diff = torch.abs(preds - y)

        correct_exact = (diff == 0).sum()
        correct_tolerance = (diff == 1).sum()
        total = y.shape[0]

        score = (correct_exact * 1.0 + correct_tolerance * 0.5) / total

        self.log("val_loss", loss)
        self.log("val_score", score)
        self.val_loss = loss
        self.val_score = score

    def on_train_epoch_end(self):
        print(f"[TRAIN] Epoch {self.current_epoch+1} | loss: {self.train_loss:.4f}")

    def on_validation_epoch_end(self):
        print(f"[VAL] Epoch {self.current_epoch+1} | loss: {self.val_loss:.4f} | score: {self.val_score:.4f}")

    def configure_optimizers(self):
        optimizer = optim.AdamW(
            self.parameters(), lr=self.learning_rate, weight_decay=1e-2
        )

        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=3
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
            },
        }

In [8]:
lightning_model = CategoryCountLightningModule()

checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
)

early_stop_callback = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=5,
)

trainer = L.Trainer(
    max_epochs=20,
    log_every_n_steps=1,
    callbacks=[checkpoint_callback, early_stop_callback],
    enable_progress_bar=True,
    enable_model_summary=True,
)

torch.set_float32_matmul_precision("high")
trainer.fit(lightning_model, train_loader, val_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type    | Params | Mode 
----------------------------------------------
0 | model     | ResNet  | 11.3 M | train
1 | criterion | MSELoss | 0      | train
----------------------------------------------
11.3 M    Trainable params
0         Non-trainable params
11.3 M    Total params
45.232    Total estimated model params size (MB)
73        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

[VAL] Epoch 1 | loss: 24.7273 | score: 0.0000


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

[VAL] Epoch 13 | loss: 0.8193 | score: 0.7188
[TRAIN] Epoch 13 | loss: 0.2442


Validation: |          | 0/? [00:00<?, ?it/s]

[VAL] Epoch 14 | loss: 0.7325 | score: 0.7188
[TRAIN] Epoch 14 | loss: 0.2469


Validation: |          | 0/? [00:00<?, ?it/s]

[VAL] Epoch 15 | loss: 0.7223 | score: 0.7500
[TRAIN] Epoch 15 | loss: 0.2130


# Submission

In [9]:
df_test = pd.read_csv(os.path.join(root_path, "test.csv"))
test_dataset = CategoryCountDataset(
    df_test, root_path, transform=val_transform, is_test=True
)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [10]:
best_model_path = checkpoint_callback.best_model_path
best_lightning_model = CategoryCountLightningModule.load_from_checkpoint(
    best_model_path
)
best_lightning_model.eval()
print("ok.")

ok.


In [11]:
predictions = []

with torch.no_grad():
    for images, sample_ids in tqdm(test_loader):
        images = images.to(device)
        outputs = best_lightning_model(images).squeeze()
        preds = torch.round(outputs).cpu().numpy()

        for sample_id, pred in zip(sample_ids, preds):
            predictions.append(
                {"SampleID": int(sample_id), "PredictedLabel": int(max(1, pred))}
            )

100%|██████████| 2/2 [00:00<00:00, 11.43it/s]


In [12]:
submission_df = pd.DataFrame(predictions)
submission_df.to_csv(os.path.join(root_path, "submission.csv"), index=False)

submission_df.head()

,SampleID,PredictedLabel
0,362,4
1,74,2
2,375,4
3,156,5
4,105,6
